In [ ]:
!pip install -q openai pandas tqdm pyarrow

In [ ]:
import os
import json
import time
import pandas as pd
from tqdm import tqdm
from getpass import getpass
from google.colab import drive
from openai import AzureOpenAI

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
os.environ["AZURE_OPENAI_API_KEY"] = getpass("Paste AZURE_OPENAI_API_KEY: ").strip()
os.environ["ENDPOINT_URL"] = getpass("Paste ENDPOINT_URL: ").strip()
os.environ["API_VERSION"] = getpass("Paste API_VERSION: ").strip()

client = AzureOpenAI(
    azure_endpoint=os.getenv("ENDPOINT_URL"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version= os.getenv("API_VERSION"),
)

GPT_MODEL = "gpt-5-mini"

In [ ]:
# ============================================================
# Paths and config
# ============================================================

INPUT_DIR = "data/intermediate/topic_candidates"
OUTPUT_DIR = "data/intermediate/topic_top8000_relevance_stance_labeled"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TOPIC_FILES = {
    "blm": "BLM_candidates.csv",
    "guns": "guns_candidates.csv",
    "samesex": "same_sex_marriage_candidates.csv",
}

TEXT_COL = "comment_text"  # change to "message" if your CSV uses message
TOP_N = 8000
BATCH_SIZE = 50
MIN_HNSW_SIMILARITY = 0.35

In [ ]:
# ============================================================
# Select top 8000 most relevant rows before GPT
# ============================================================

def select_top_relevant_rows(df, top_n=8000):
    df = df.copy()

    if {"keyword_match", "hnsw_similarity", "topic_candidate_score"}.issubset(df.columns):
        df["keyword_match"] = df["keyword_match"].astype(str).str.lower().isin(["true", "1", "yes"])
        df["hnsw_similarity"] = pd.to_numeric(df["hnsw_similarity"], errors="coerce").fillna(0.0)
        df["topic_candidate_score"] = pd.to_numeric(df["topic_candidate_score"], errors="coerce").fillna(0.0)

        filtered = df[
            (df["keyword_match"] == True) |
            (df["hnsw_similarity"] >= MIN_HNSW_SIMILARITY)
        ].copy()

        if len(filtered) < top_n:
            print(f"Only {len(filtered)} rows passed the strict filter, so using score ranking from all rows.")
            filtered = df.copy()

        filtered = filtered.sort_values(
            ["topic_candidate_score", "hnsw_similarity"],
            ascending=False
        ).head(top_n).copy()

        return filtered

    if "topic_candidate_score" in df.columns:
        df["topic_candidate_score"] = pd.to_numeric(df["topic_candidate_score"], errors="coerce").fillna(0.0)
        return df.sort_values("topic_candidate_score", ascending=False).head(top_n).copy()

    print("WARNING: No topic_candidate_score found. Taking first rows only.")
    return df.head(top_n).copy()

In [ ]:
def topic_definition(topic):
    if topic == "blm":
        return """
Topic: Black Lives Matter.

A comment is relevant only if it discusses Black Lives Matter, racial justice, systemic racism, police brutality against Black people, George Floyd, racial inequality, or direct arguments about BLM.

pro = supports BLM or the racial justice/systemic racism framing.
anti = criticizes BLM, rejects the racial justice/systemic racism framing, calls BLM divisive/racist/useless/violent, or supports All Lives Matter as a rejection of BLM.
neutral_or_mixed = relevant to BLM but unclear, balanced, factual, or both sides.
irrelevant = not actually about BLM.
"""

    if topic == "guns":
        return """
Topic: Gun control.

A comment is relevant only if it discusses gun control, firearm laws, gun rights, the Second Amendment, background checks, assault weapons, NRA, gun bans, open carry, concealed carry, or gun ownership policy.

pro = supports stricter gun control, restrictions, bans, background checks, or reducing gun access.
anti = opposes gun control, supports gun rights, Second Amendment framing, firearm ownership, or argues gun restrictions are harmful/useless.
neutral_or_mixed = relevant to gun control but unclear, balanced, factual, or both sides.
irrelevant = mentions guns only incidentally or is not about gun policy.
"""

    if topic == "samesex":
        return """
Topic: Same-sex marriage legalization.

A comment is relevant only if it discusses same-sex marriage, gay marriage, marriage equality, civil unions, or legal recognition of same-sex couples.

pro = supports same-sex marriage, marriage equality, LGBT marriage rights, or equal legal recognition.
anti = opposes same-sex marriage, rejects marriage equality, or supports limiting marriage to heterosexual couples.
neutral_or_mixed = relevant to same-sex marriage but unclear, balanced, factual, or both sides.
irrelevant = about LGBT issues generally but not same-sex marriage, or not about this topic.
"""

    raise ValueError(topic)

In [ ]:
def build_label_prompt(topic, batch_df, text_col):
    rows = []

    for _, row in batch_df.iterrows():
        text = str(row[text_col]) if pd.notna(row[text_col]) else ""
        text = text.replace("\n", " ").strip()

        rows.append({
            "id": int(row["_label_local_id"]),
            "text": text[:1500],
        })

    return f"""
You are labeling political comments for a research dataset.

{topic_definition(topic)}

Classify each comment into exactly one label:
pro, anti, neutral_or_mixed, irrelevant.

Important:
First decide whether the comment is actually relevant to the topic.
If it is not clearly about the topic, label it irrelevant.
Do not infer relevance only because one keyword appears.
Do not classify based on toxicity or politeness.
Focus only on the political stance toward the topic.

Return valid JSON only in this exact format:
{{
  "items": [
    {{
      "id": 0,
      "stance_label": "pro",
      "confidence": 0.95,
      "short_reason": "brief explanation"
    }}
  ]
}}

Comments:
{json.dumps(rows, ensure_ascii=False)}
"""

In [ ]:
def extract_json_from_text(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(text[start:end + 1])
        raise

In [ ]:
def classify_batch(topic, batch_df, text_col, max_retries=4):
    prompt = build_label_prompt(topic, batch_df, text_col)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=AZURE_DEPLOYMENT_NAME,
                messages=[
                    {
                        "role": "system",
                        "content": "You are a careful political stance classifier. Return valid JSON only."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
            )

            content = response.choices[0].message.content
            parsed = extract_json_from_text(content)

            return parsed.get("items", [])

        except Exception as e:
            print(f"Attempt {attempt + 1} failed:", repr(e))
            time.sleep(5 + attempt * 5)

    return []

In [ ]:
def classify_topic_file(topic, input_path, output_dir, text_col):
    print("\n" + "=" * 80)
    print(f"Processing topic: {topic}")
    print("=" * 80)

    df = pd.read_csv(input_path, encoding="utf-8-sig")

    if text_col not in df.columns:
        raise ValueError(f"Column {text_col} not found. Columns: {df.columns.tolist()}")

    print(f"Original rows: {len(df):,}")

    df = select_top_relevant_rows(df, top_n=TOP_N).copy()

    print(f"Rows selected for GPT labeling: {len(df):,}")

    top8000_path = os.path.join(output_dir, f"{topic}_top{len(df)}_before_gpt.csv")
    df.to_csv(top8000_path, index=False, encoding="utf-8-sig")
    print("Saved top relevant rows:", top8000_path)

    df = df.reset_index(drop=True).copy()
    df["_label_local_id"] = range(len(df))

    all_items = []

    for start in tqdm(range(0, len(df), BATCH_SIZE), desc=f"GPT labeling {topic}"):
        batch = df.iloc[start:start + BATCH_SIZE]
        items = classify_batch(topic, batch, text_col)
        all_items.extend(items)
        time.sleep(0.5)

    labels_df = pd.DataFrame(all_items)

    if labels_df.empty:
        labels_df = pd.DataFrame(columns=["id", "stance_label", "confidence", "short_reason"])

    rename_map = {
        "stance": "stance_label",
        "label": "stance_label",
        "classification": "stance_label",
        "reason": "short_reason",
        "rationale": "short_reason",
    }

    labels_df = labels_df.rename(columns={
        old: new for old, new in rename_map.items() if old in labels_df.columns
    })

    for col, default in {
        "id": None,
        "stance_label": "irrelevant",
        "confidence": 0.0,
        "short_reason": "missing_or_parse_error",
    }.items():
        if col not in labels_df.columns:
            labels_df[col] = default

    labels_df["id"] = pd.to_numeric(labels_df["id"], errors="coerce")
    labels_df["confidence"] = pd.to_numeric(labels_df["confidence"], errors="coerce")

    labels_df = labels_df.dropna(subset=["id"])
    labels_df["id"] = labels_df["id"].astype(int)

    df_clean = df.drop(
        columns=[
            "id",
            "stance_label",
            "confidence",
            "short_reason",
            "stance",
            "label",
            "classification",
            "reason",
            "rationale",
        ],
        errors="ignore"
    )

    merged = df_clean.merge(
        labels_df[["id", "stance_label", "confidence", "short_reason"]],
        left_on="_label_local_id",
        right_on="id",
        how="left"
    )

    merged["stance_label"] = merged["stance_label"].fillna("irrelevant")
    merged["confidence"] = merged["confidence"].fillna(0.0)
    merged["short_reason"] = merged["short_reason"].fillna("missing_or_parse_error")

    allowed = {"pro", "anti", "neutral_or_mixed", "irrelevant"}
    merged.loc[~merged["stance_label"].isin(allowed), "stance_label"] = "irrelevant"

    labeled_path = os.path.join(output_dir, f"{topic}_top{TOP_N}_all_labeled.csv")
    merged.to_csv(labeled_path, index=False, encoding="utf-8-sig")

    clean_pro_anti = merged[
        (merged["stance_label"].isin(["pro", "anti"])) &
        (merged["confidence"] >= 0.60)
    ].copy()

    clean_path = os.path.join(output_dir, f"{topic}_clean_pro_anti.csv")
    clean_pro_anti.to_csv(clean_path, index=False, encoding="utf-8-sig")

    print("\nSaved labeled file:", labeled_path)
    print("Saved clean pro/anti file:", clean_path)

    print("\nAll label counts:")
    print(merged["stance_label"].value_counts())

    print("\nClean pro/anti counts:")
    print(clean_pro_anti["stance_label"].value_counts())

    return merged, clean_pro_anti

In [ ]:
all_labeled = {}
all_clean = {}

for topic, filename in TOPIC_FILES.items():
    input_path = os.path.join(INPUT_DIR, filename)

    labeled_df, clean_df = classify_topic_file(
        topic=topic,
        input_path=input_path,
        output_dir=OUTPUT_DIR,
        text_col=TEXT_COL,
    )

    all_labeled[topic] = labeled_df
    all_clean[topic] = clean_df